# Suite JRN — Journals

A journal is mbse-schemas' `Changes.Journal`, as plain data: its transactions, in order, each with its unique `id`, as a store's `follow` gives them, a journal of one transaction at a time. It is data, kept in no store: mbse-journals takes the whole transactions after a position, a transaction's id, joins journals and inserts one after a position, splits one into the one-transaction journals a stream carries and joins them back, and writes, appends and reads it in a file of JSON lines; a store holding the same schemas applies it (`Changes.apply`). mbse-basis [the journals proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/JOURNALS.md#decided), [CAP-273: journals in memory and in local files](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-273) and [CAP-274: journals sent and received](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-274).

A store appears only where a journal is given (`store.follow`) and where one is applied (`Changes.apply`), each made by `made` alone, so that the suite runs over any store given in its place: the journals themselves are never a store's.

In [ ]:
import { mkdtempSync, readFileSync, writeFileSync, existsSync } from "node:fs";
import { tmpdir } from "node:os";
import { join } from "node:path";

import { Builders, Changes, JSON as Json, Plain } from "@mbse/schemas/Framework";
import { DecodeError, LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import type { PlainData, PlainMap } from "@mbse/schemas/Framework/Plain";

import { Journals } from "../src/Framework/index.js";

import { assert } from "./support.js";

type Journal = Journals.Journal;

/** crm.Person, owned by the singleton book by key, with a name. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Holds: S.OfRelation, Book: S.OfObject, Person: S.OfObject })
    .spec([
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")])
        .properties({ key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr }).adjacencies([
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true)]),
    ])]))]);
}

/** A store holding the schemas `install` defines: the one place the suite makes a store. */
function made(install: (txn: any) => any): any {
  const store: any = new Builders.Store();
  store.transaction(install);
  return store;
}

/** A transaction of `store` adding a person to its book by each key, named by it. */
function add(store: any, ...keys: string[]): void {
  store.transaction(() => {
    const book = store.singleton("crm.TheBook");
    for (const key of keys) {
      store.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key(key).book(book))).name(key));
    }
  });
}

/** A transaction of `store` naming the person its book holds by `key`. */
function rename(store: any, key: string, name: string): void {
  store.transaction(() => {
    store.crm.Person(store.singleton("crm.TheBook").contacts({ key }), (p: any) => p.name(name));
  });
}

/** A transaction of `store` removing the person its book holds by `key`. */
function remove(store: any, key: string): void {
  store.transaction(() => {
    store.crm.Book(store.singleton("crm.TheBook"), (b: any) => b.contacts((a: any) => a.remove((e: any) => e.key(key))));
  });
}

/** The book's contacts, each its key and name, by key, as JSON text. */
function contacts(store: any): string {
  let held: PlainData[] = [];
  store.transaction(() => {
    held = [...store.singleton("crm.TheBook").contacts].map((e: any): PlainData =>
      [e.key.get(), e.contact.name.get() ?? null]).sort((a: any, b: any) => a[0] < b[0] ? -1 : 1);
  });
  return Json.dumps(held);
}

/** `store`, a store holding the crm schemas made where none is given, with `journal` applied to it at once: read as
 * a `Changes.Journal`, a value of the store's, and applied in one transaction (`Changes.apply`). */
function applied(journal: Journal, store: any = null): any {
  store = store ?? made(crm);
  const value = Plain.FromPlain(store)(store.registered("Changes.Journal"), journal);
  store.transaction((txn: any) => Changes.apply(txn, value));
  return store;
}

/** A journal's positions: its transactions' ids, in order. */
function ids(journal: Journal): string[] {
  return journal.map((transaction) => transaction.get("id") as string);
}

/** Whether two journals, or any plain data, are the same: the same JSON text. */
function same(a: PlainData, b: PlainData): boolean {
  return Json.dumps(a) === Json.dumps(b);
}

/** The message of the `error` `call` throws. */
function refused(error: new (...args: any[]) => Error, call: () => unknown): string {
  try {
    call();
  } catch (raised) {
    if (raised instanceof error) return raised.message;
    throw raised;
  }
  throw new Error("not refused");
}

/** A transaction, as plain data, from its members. */
function map(members: Record<string, PlainData>): PlainMap {
  return new Map(Object.entries(members));
}

<a id="JRN-01"></a>
## JRN-01 · A store's follow, collected, builds a journal: each transaction the store commits, in order, once, as plain data, each with its id, a position; once the follow ends, none is given

In [ ]:
const model = made(crm);
const journal: Journal = [];
const end = model.follow(Journals.collect(journal));
add(model, "ada", "bob");
add(model, "cy");
rename(model, "ada", "Ada");
remove(model, "bob");
end();
const [first, second, third, fourth] = ids(journal) as [string, string, string, string];
assert(new Set(ids(journal)).size === 4 && ids(journal).every((i) => typeof i === "string" && i !== ""));
assert(same(Json.loads(Json.dumps(journal)), journal)); // plain data
assert(Json.dumps(journal[2]!.get("changes")!) === '[{"property": {"holder": "o0", "name": "name", "mutations": [{"set": {"value": "Ada"}}]}}]');
assert(Json.dumps(journal[3]!.get("changes")!) === '[{"entry": {"holder": {"reference": {"symbol": "o0"}}, "adjacency": "contacts", "verb": "remove", "links": {}, "properties": {"key": "bob"}}}]');
const other = made(crm);
const given: Journal = [];
other.follow(Journals.collect(given))(); // ended at once
add(other, "zed");
assert(given.length === 0);

<a id="JRN-02"></a>
## JRN-02 · The whole transactions after a position are a journal, up to and including one where given, from the first where none is; after the last, none; a position the journal lacks is refused, and the journal sliced is unchanged

In [ ]:
assert(Json.dumps(ids(Journals.after(journal, first))) === Json.dumps([second, third, fourth]));
assert(Json.dumps(ids(Journals.after(journal, first, third))) === Json.dumps([second, third]));
assert(Json.dumps(ids(Journals.after(journal, null, second))) === Json.dumps([first, second]));
assert(same(Journals.after(journal), journal) && Journals.after(journal, fourth).length === 0);
assert(refused(LookupError, () => Journals.after(journal, "t0")) === 'the journal holds no transaction "t0"');
assert(refused(LookupError, () => Journals.after(journal, third, first)) === `the journal holds no transaction ${Json.dumps(first)}`);
assert(Json.dumps(ids(journal)) === Json.dumps([first, second, third, fourth]));

<a id="JRN-03"></a>
## JRN-03 · Journals joined are one journal of their transactions, in order; a transaction arriving again is held once, in its place, the later arrival replacing it; a journal inserted after a position, or before the first, is read in its place; a position the journal lacks is refused

In [ ]:
assert(same(Journals.join(journal.slice(0, 2), journal.slice(2)), journal) && Journals.join().length === 0);
const again = new Map([...journal[1]!, ["changes", []]]); // the second, arriving again, other than it was
const joined = Journals.join(journal.slice(0, 3), [again], journal.slice(2));
assert(same(ids(joined), ids(journal)) && same(joined[1]!.get("changes")!, []) && !same(journal[1]!.get("changes")!, []));
assert(same(Journals.insert([...journal.slice(0, 1), ...journal.slice(2)], first, journal.slice(1, 2)), journal));
assert(same(Journals.insert(journal.slice(1), null, journal.slice(0, 1)), journal));
assert(same(Journals.insert(journal, fourth, []), journal));
assert(same(ids(Journals.insert(journal, first, journal.slice(3))), [first, fourth, second, third])); // moved: held once
assert(refused(LookupError, () => Journals.insert(journal, "t0", journal)) === 'the journal holds no transaction "t0"');

<a id="JRN-04"></a>
## JRN-04 · A journal split into its one-transaction journals, as a follow gives them, each sent as JSON text, is rebuilt by joining them; a peer holding the journal up to a position, sent the slice after an earlier one, catches up, holding each transaction once

In [ ]:
const sent = Journals.split(journal).map((one) => Json.dumps(one));
assert(sent.length === 4 && sent.every((text, i) => same(ids(Json.loads(text) as Journal), [ids(journal)[i]!])));
assert(same(Journals.join(...sent.map((text) => Json.loads(text) as Journal)), journal));
const peer = journal.slice(0, 2); // the peer holds the journal up to the second
const receive = Journals.collect(peer);
for (const one of Journals.split(Journals.after(journal, first))) { // sent from the first: the second arrives again
  receive(Json.loads(Json.dumps(one)));
}
assert(same(peer, journal));

<a id="JRN-05"></a>
## JRN-05 · A journal written to a file is one line for each transaction, its JSON text; whole transactions are appended to it, or to one made where there is none; read back, it is the journal, or the part after a position, a transaction appended again read once; a last line not ended is read, and one cut short is refused

In [ ]:
const folder = mkdtempSync(join(tmpdir(), "jrn-"));
const [path, madeByAppend, cut] = ["crm.jsonl", "new.jsonl", "cut.jsonl"].map((name) => join(folder, name)) as [string, string, string];
Journals.write(path, journal.slice(0, 2));
assert(readFileSync(path, "utf8") === Json.dumps(journal[0]!) + "\n" + Json.dumps(journal[1]!) + "\n");
Journals.append(path, journal.slice(2));
assert(same(Journals.read(path), journal) && same(Journals.read(path, second), journal.slice(2)));
Journals.append(path, journal.slice(3)); // the last, again
assert(same(Journals.read(path), journal) && readFileSync(path, "utf8").split("\n").length === 6);
Journals.append(madeByAppend, journal.slice(0, 1));
assert(same(Journals.read(madeByAppend), journal.slice(0, 1)));
Journals.write(madeByAppend, []);
assert(readFileSync(madeByAppend, "utf8") === "" && Journals.read(madeByAppend).length === 0);
writeFileSync(cut, Json.dumps(journal[0]!), "utf8"); // no line end
assert(same(Journals.read(cut), journal.slice(0, 1)));
writeFileSync(cut, Json.dumps(journal[0]!) + "\n" + Json.dumps(journal[1]!).slice(0, 20), "utf8"); // a write cut short
assert(refused(DecodeError, () => Journals.read(cut)) !== "");
assert(refused(LookupError, () => Journals.read(path, "t0")) === 'the journal holds no transaction "t0"');

<a id="JRN-06"></a>
## JRN-06 · A journal, as plain data or read from a file, applied to a store holding the same schemas (`Changes.apply`), makes it hold what the followed store does, and applied again changes nothing; the slice after a position, applied to a store holding the journal up to it, makes it the same

In [ ]:
const replica = applied(journal);
assert(contacts(replica) === contacts(model) && contacts(model) === '[["ada", "Ada"], ["cy", "cy"]]');
applied(Journals.read(path), replica);
assert(contacts(replica) === contacts(model));
const behind = applied(Journals.after(journal, null, second));
assert(contacts(behind) === '[["ada", "ada"], ["bob", "bob"], ["cy", "cy"]]');
applied(Journals.after(journal, second), behind);
assert(contacts(behind) === contacts(model));

<a id="JRN-07"></a>
## JRN-07 · What is not a journal as plain data, a list of its transactions each with its id, is refused by each function, and nothing is written

In [ ]:
const badJournals: any[] = [map({ id: "t1" }), [map({ changes: [] })], [map({ id: 1 })], ["t1"]];
const message = "a journal, as plain data, is a list of its transactions, each with its id";
const calls: ((bad: any) => unknown)[] = [(bad) => Journals.after(bad), (bad) => Journals.join(journal, bad),
  (bad) => Journals.insert(bad, null, journal), (bad) => Journals.insert(journal, null, bad), Journals.split,
  Journals.collect, (bad) => Journals.collect([...journal])(bad), (bad) => Journals.write(join(folder, "bad.jsonl"), bad),
  (bad) => Journals.append(join(folder, "bad.jsonl"), bad)];
assert(calls.every((call) => badJournals.every((bad) => refused(ValueError, () => call(bad)) === message)));
assert(!existsSync(join(folder, "bad.jsonl")));
writeFileSync(join(folder, "bad.jsonl"), '"t1"\n', "utf8");
assert(refused(ValueError, () => Journals.read(join(folder, "bad.jsonl"))) === message);